# L6: 监督学习算法


# L6: 监督学习算法

**课时**: 2 课时（90 分钟/课时）

## 1. 学习目标

完成本课程后，学员能够：
1. 推导线性回归与逻辑回归的损失函数与梯度更新公式
2. 理解决策树的分裂准则（信息熵、基尼系数）与剪枝策略
3. 掌握SVM的间隔最大化原理与核函数技巧
4. 使用sklearn实现完整的监督学习 pipeline
5. 理解模型评估指标（准确率、精确率、召回率、F1、AUC）

## 2. 核心知识点

### 2.1 线性回归

**目标**: 最小化均方误差 MSE = (1/n)∑(yᵢ - ŷᵢ)²

**解析解**: θ = (XᵀX)⁻¹Xᵀy （当 XᵀX 可逆时）

**梯度下降**: θ := θ - α·Xᵀ(Xθ - y)/n


In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.datasets import make_regression
from sklearn.model_selection import cross_val_score

X, y = make_regression(n_samples=1000, n_features=20, noise=10, random_state=42)

# 解析解
from numpy.linalg import inv
X_b = np.c_[np.ones(X.shape[0]), X]
theta_best = inv(X_b.T @ X_b) @ X_b.T @ y

# 梯度下降
sgd = SGDRegressor(max_iter=1000, tol=1e-3, random_state=42)
sgd.fit(X, y)

# sklearn API
lr = LinearRegression()
cv_scores = cross_val_score(lr, X, y, cv=5, scoring='r2')
print(f"5折交叉验证 R²: {cv_scores.mean():.3f}")


### 2.2 逻辑回归

**激活函数**: σ(z) = 1/(1+e⁻ᵗ) （Sigmoid）

**损失函数**: 二元交叉熵 BCE = -[y·log(ŷ) + (1-y)·log(1-ŷ)]


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report

data = load_breast_cancer()
X, y = data.data, data.target

lr = LogisticRegression(max_iter=1000, C=1.0)
cv_scores = cross_val_score(lr, X, y, cv=5, scoring='accuracy')
print(f"5折准确率: {cv_scores.mean():.3f}")

y_pred = lr.predict(X)
print(classification_report(y, y_pred, target_names=data.target_names))


### 2.3 决策树

**分裂准则**:
- **信息熵**: H(S) = -∑pᵢlog₂(pᵢ)
- **基尼系数**: Gini(S) = 1-∑pᵢ²

**信息增益**: IG(S, A) = H(S) - ∑(|Sᵥ|/|S|)·H(Sᵥ)


In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.datasets import load_iris

iris = load_iris()
X, y = iris.data, iris.target

dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X, y)

fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(dt, feature_names=iris.feature_names, class_names=iris.target_names,
          filled=True, rounded=True, ax=ax)
plt.savefig("l6_decision_tree.png", dpi=150, bbox_inches='tight')


### 2.4 支持向量机（SVM）

**核心思想**: 间隔最大化 → 找到最大间隔超平面

**核函数技巧**: 将数据映射到高维空间使其线性可分
- 线性核: K(x, z) = x·z
- RBF核（高斯核）: K(x, z) = exp(-γ||x-z||²) — 最常用


In [ ]:
from sklearn.svm import SVC
from sklearn.datasets import make_circles

X, y = make_circles(n_samples=200, factor=0.3, noise=0.1, random_state=42)

svm_rbf = SVC(kernel='rbf', C=1.0, gamma='scale')
svm_rbf.fit(X, y)
print(f"RBF-SVM准确率: {svm_rbf.score(X, y):.2%}")


### 2.5 模型评估指标

| 指标 | 公式 | 适用场景 |
|------|------|----------|
| 准确率 (Accuracy) | (TP+TN)/(TP+TN+FP+FN) | 类别平衡 |
| 精确率 (Precision) | TP/(TP+FP) | 假阳性代价高 |
| 召回率 (Recall) | TP/(TP+FN) | 假阴性代价高 |
| F1 Score | 2·P·R/(P+R) | 综合精确与召回 |
| AUC-ROC | ROC曲线下面积 | 不平衡数据集 |


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(f"准确率: {accuracy_score(y_test, y_pred):.3f}")
print(f"精确率: {precision_score(y_test, y_pred):.3f}")
print(f"召回率: {recall_score(y_test, y_pred):.3f}")
print(f"F1分数: {f1_score(y_test, y_pred):.3f}")
print(f"AUC-ROC: {roc_auc_score(y_test, y_proba):.3f}")


## 3. 代码示例


In [ ]:
"""
L6-supervised-learning.py
综合演示：监督学习算法对比实验
"""

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

X, y = make_moons(n_samples=500, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

models = {
    "逻辑回归": LogisticRegression(),
    "决策树(深)": DecisionTreeClassifier(max_depth=10, random_state=42),
    "决策树(浅)": DecisionTreeClassifier(max_depth=3, random_state=42),
    "SVM(RBF)": SVC(kernel='rbf', C=1.0),
    "随机森林": RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    cv_scores = cross_val_score(model, X, y, cv=5)
    results[name] = {"train": train_acc, "test": test_acc, "cv": cv_scores.mean()}
    print(f"{name}: 训练={train_acc:.3f}, 测试={test_acc:.3f}, CV={cv_scores.mean():.3f}")

# 性能对比柱状图
fig, ax = plt.subplots(figsize=(10, 6))
names = list(results.keys())
train_accs = [results[n]["train"] for n in names]
test_accs = [results[n]["test"] for n in names]
x = np.arange(len(names))
width = 0.35
ax.bar(x - width/2, train_accs, width, label='训练准确率', color='#3498db')
ax.bar(x + width/2, test_accs, width, label='测试准确率', color='#e74c3c')
ax.set_ylabel('准确率')
ax.set_title('模型性能对比')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=45, ha='right')
ax.legend()
ax.set_ylim(0.5, 1.05)
plt.tight_layout()
plt.savefig("l6_model_comparison.png", dpi=150)
plt.show()


## 4. 练习题

1. **线性回归推导**: 手动推导线性回归的闭式解，并说明XᵀX不可逆的情况。
2. **逻辑回归实现**: 不使用sklearn，自己实现逻辑回归的梯度下降。
3. **决策树实验**: 在wine数据集上观察不同max_depth、min_samples_split对模型的影响。
4. **SVM调参**: 比较不同C值和核函数（linear, poly, rbf）对决策边界的影响。
5. **综合评估**: 对真实数据集完成：数据预处理→5种算法建模→计算所有指标→绘制ROC曲线→选出最优模型。

## 5. 延伸阅读

- **书籍**: Gareth James et al., *An Introduction to Statistical Learning*（第2版）
- **在线课程**: StatQuest with Josh Starmer — YouTube频道
- **工具**: Jupyter机器学习教程 — https://scikit-learn.org/stable/tutorial/index.html

---
